# Virtual Mouse / Air Canvas - Phase 3

## Cursor movement (gesture-toggled, smoothed)

Phase 3 turns the index fingertip into your real mouse cursor. Three new ideas:

1. **Gesture toggle** - show an **open palm** to ENABLE control, a **fist** to
   DISABLE it. The cursor does nothing until you enable it.
2. **Interaction window** - your hand moves inside a small box in the centre of
   the camera view, which maps to the whole screen. No need to sweep your arm.
3. **EMA smoothing** - an exponential moving average filter removes the jitter
   so the cursor glides instead of shaking.

---
### Safety - read before running
- The cursor only moves while control is **enabled** (open palm to turn on).
- Make a **fist** any time to freeze it instantly.
- PyAutoGUI failsafe stays ON: **slam your real mouse into a screen corner**
  to abort the whole loop immediately.
- Press **`q`** in the camera window to quit.

Run cells top to bottom.

## 1. Imports

In [1]:
import time
import platform
import threading
from collections import deque

import cv2
import numpy as np
import mediapipe as mp
import pyautogui

assert hasattr(mp.solutions, "hands"), "Need mediapipe==0.10.21"
pyautogui.FAILSAFE = True      # mouse to a screen corner = abort
pyautogui.PAUSE = 0            # we manage our own timing
SCREEN_W, SCREEN_H = pyautogui.size()
print("Imports OK. Screen:", SCREEN_W, "x", SCREEN_H)

Imports OK. Screen: 1920 x 1080


## 2. Threaded camera
Same as Phase 2 - background thread, one-slot buffer, always read the newest frame.

In [2]:
class ThreadedCamera:
    def __init__(self, src=0, width=640, height=480):
        if platform.system() == "Windows":
            self.cap = cv2.VideoCapture(src, cv2.CAP_DSHOW)
        else:
            self.cap = cv2.VideoCapture(src)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        self.buffer = deque(maxlen=1)
        self.running = False
        self.lock = threading.Lock()
        self.thread = None

    def opened(self):
        return self.cap.isOpened()

    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()
        return self

    def _loop(self):
        while self.running:
            ok, frame = self.cap.read()
            if ok:
                with self.lock:
                    self.buffer.append(frame)

    def read(self):
        with self.lock:
            return self.buffer[-1] if self.buffer else None

    def stop(self):
        self.running = False
        if self.thread is not None and self.thread.is_alive():
            self.thread.join(timeout=1)
        self.cap.release()

## 3. Hand tracker + finger-state detection

`fingers_up()` returns a list like `[1,1,0,0,0]` (thumb, index, middle, ring,
pinky). A finger counts as "up" when its tip is higher than its middle joint.
The thumb is judged horizontally because it bends sideways.

From this we read simple poses:
- **open palm** = all five up  -> enable control
- **fist** = none up           -> disable control
- **pointing** = index up only -> move the cursor

In [3]:
WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP, RING_TIP, PINKY_TIP = 4, 8, 12, 16, 20
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]   # joint below each tip


class HandTracker:
    def __init__(self, max_hands=1, detection_conf=0.7, tracking_conf=0.5):
        self.mp_hands = mp.solutions.hands
        self.mp_draw = mp.solutions.drawing_utils
        self.mp_styles = mp.solutions.drawing_styles
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=detection_conf,
            min_tracking_confidence=tracking_conf,
        )

    def process(self, frame_bgr):
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = self.hands.process(rgb)
        rgb.flags.writeable = True
        return results

    def draw(self, frame_bgr, results):
        if results.multi_hand_landmarks:
            for lm in results.multi_hand_landmarks:
                self.mp_draw.draw_landmarks(
                    frame_bgr, lm, self.mp_hands.HAND_CONNECTIONS,
                    self.mp_styles.get_default_hand_landmarks_style(),
                    self.mp_styles.get_default_hand_connections_style(),
                )

    @staticmethod
    def landmark_pixel(hand_landmarks, index, frame_shape):
        h, w = frame_shape[:2]
        lm = hand_landmarks.landmark[index]
        return int(lm.x * w), int(lm.y * h)

    @staticmethod
    def fingers_up(hand_landmarks):
        lm = hand_landmarks.landmark
        fingers = []
        # Thumb: compare x of tip vs joint (sideways bend). Works for a right
        # hand in a mirrored frame; good enough for open-palm / fist poses.
        fingers.append(1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0)
        # Other four: tip higher (smaller y) than its PIP joint = up
        for tip, pip in zip(TIPS[1:], PIPS[1:]):
            fingers.append(1 if lm[tip].y < lm[pip].y else 0)
        return fingers

    def close(self):
        self.hands.close()

## 4. EMA smoothing filter

Raw fingertip coordinates jitter from sensor noise. The exponential moving
average blends each new point with the previous smoothed point:

`smoothed = alpha * new + (1 - alpha) * previous`

Lower `alpha` = smoother but more lag. Higher = snappier but shakier.
Start at **0.3** and tune to taste.

In [4]:
class EMAFilter:
    def __init__(self, alpha=0.3):
        self.alpha = alpha
        self.x = None
        self.y = None

    def filter(self, x, y):
        if self.x is None:
            self.x, self.y = x, y
        else:
            self.x = self.alpha * x + (1 - self.alpha) * self.x
            self.y = self.alpha * y + (1 - self.alpha) * self.y
        return self.x, self.y

    def reset(self):
        self.x = self.y = None

## 5. Tuning knobs

`ACTIVE_*` define the interaction box inside the camera frame (as fractions,
so they scale with resolution). Only this central box maps to your screen,
which keeps movement comfortable and avoids the noisy frame edges.

In [9]:
ALPHA = 0.3
ACTIVE_LEFT, ACTIVE_RIGHT = 0.30, 0.70
ACTIVE_TOP, ACTIVE_BOTTOM = 0.25, 0.65

EDGE_MARGIN = 3   # keep cursor this many px away from screen edges

def map_to_screen(nx, ny):
    """Map a normalized fingertip to screen pixels, staying just inside the
    edges so the cursor never lands exactly on a corner (which would trip
    PyAutoGUI's failsafe during normal use)."""
    sx = np.interp(nx, (ACTIVE_LEFT, ACTIVE_RIGHT), (EDGE_MARGIN, SCREEN_W - EDGE_MARGIN))
    sy = np.interp(ny, (ACTIVE_TOP, ACTIVE_BOTTOM), (EDGE_MARGIN, SCREEN_H - EDGE_MARGIN))
    return sx, sy

## 6. Run - gesture-toggled cursor

- **Open palm** -> control ON (status turns green)
- **Fist** -> control OFF
- **Point** (index only) while ON -> cursor follows your fingertip
- **`q`** in the window -> quit | mouse to a screen corner -> hard abort

In [1]:
camera = ThreadedCamera(src=0)
if not camera.opened():
    print("ERROR: could not open camera. Close other camera apps / try src=1.")
else:
    camera.start()
    time.sleep(1.0)
    tracker = HandTracker(max_hands=1)
    ema = EMAFilter(alpha=ALPHA)

    control_enabled = False
    prev = time.time()
    print("Running. Open palm = ON, fist = OFF, point to move, 'q' to quit.")

    try:
        while True:
            frame = camera.read()
            if frame is None:
                continue
            frame = cv2.flip(frame, 1)
            h, w = frame.shape[:2]
            results = tracker.process(frame)
            tracker.draw(frame, results)

            # draw the active interaction box
            x1, y1 = int(ACTIVE_LEFT * w), int(ACTIVE_TOP * h)
            x2, y2 = int(ACTIVE_RIGHT * w), int(ACTIVE_BOTTOM * h)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 200, 0), 2)

            if results.multi_hand_landmarks:
                hand = results.multi_hand_landmarks[0]
                fingers = tracker.fingers_up(hand)
                total = sum(fingers)

                if total == 5:
                    control_enabled = True
                elif total == 0:
                    control_enabled = False
                    ema.reset()

                # pointing = index up, middle down -> move cursor
                pointing = fingers[1] == 1 and fingers[2] == 0
                if control_enabled and pointing:
                    tip = hand.landmark[INDEX_TIP]
                    sx, sy = map_to_screen(tip.x, tip.y)
                    sx, sy = ema.filter(sx, sy)
                    sx = float(np.clip(sx, EDGE_MARGIN, SCREEN_W - EDGE_MARGIN))
                    sy = float(np.clip(sy, EDGE_MARGIN, SCREEN_H - EDGE_MARGIN))
                    pyautogui.moveTo(sx, sy)
                    cx, cy = tracker.landmark_pixel(hand, INDEX_TIP, frame.shape)
                    cv2.circle(frame, (cx, cy), 10, (0, 255, 0), cv2.FILLED)

            # status overlay
            label = "CONTROL: ON" if control_enabled else "CONTROL: OFF"
            color = (0, 200, 0) if control_enabled else (0, 0, 255)
            cv2.putText(frame, label, (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            now = time.time()
            fps = 1.0 / (now - prev) if now != prev else 0.0
            prev = now
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            cv2.imshow("Phase 3 - Cursor Control", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.stop()
        tracker.close()
        cv2.destroyAllWindows()
        print("Stopped cleanly.")

NameError: name 'ThreadedCamera' is not defined

## 7. Cleanup (run if the window hangs)

In [11]:
try:
    camera.stop()
except Exception:
    pass
cv2.destroyAllWindows()
for _ in range(4):
    cv2.waitKey(1)
print("cleaned up")

cleaned up


## Tuning tips
- Cursor too shaky? Lower `ALPHA` (e.g. 0.2). Too laggy? Raise it (e.g. 0.5).
- Cursor too sensitive / hard to reach edges? Widen the `ACTIVE_*` box.
- Cursor can't reach screen corners? Shrink the `ACTIVE_*` box.
- Thumb detection is approximate; open-palm vs fist is what matters for the
  toggle, and those are robust.

Next: **Phase 4** adds clicks, drag and scroll on top of this, using a pinch
gesture and the same finger-state logic - plus debouncing so you don't get
phantom clicks.